# Cross-Model Baseline — visualizations (per side model)

Per-round trajectories for the **cross-model baseline** (a MAIN model — Qwen2.5-14B — reads documents
synthesized from a SIDE model's answers; the MAIN model is measured via `run["answer"]`). Three side
models: **DeepSeek-R1-Distill-7B**, **Llama-3.1-8B**, **Mistral-7B**.

Figures are written **per side model** to `cross_model_baseline_visualizations/<side>/`, each overlaying
that side's `replace_all` / `replace_one` / `search` in `visualization.ipynb` style
(Replace All `#1f77b4` / Replace One `#ff7f0e` / Search `#2ca02c`).

- **Text metrics** (all 3 sides, from `evaluation.py`): cosine / ROUGE-L / TES, `same_answer%`,
  `unique_words`, `ai_reference%`, plus a combined grid. Read from
  `cross-model-baseline/evaluation_outputs/<side>/local_<variant>_eval.json` (gitignored downloads).
- **Entity metrics** (DeepSeek side only — the only cross-model entity data in the dump): `unique_entities`,
  `entity_similarity`, `collapse_by_simulation` (95% Wilson CI) for `replace_all`/`replace_one`, from the
  latest `entity re-run for workshop paper-*` dump.

In [1]:
import os
import json
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

# ── Config ──
EVAL_ROOT = "cross-model-baseline/evaluation_outputs"   # side-keyed: <side>/local_<variant>_eval.json
OUT_ROOT  = "cross_model_baseline_visualizations"
SIDES = ["deepseek-r1-distill-qwen-7b", "llama-3.1-8b", "mistral-7b"]

# label -> eval filename (missing files are skipped)
VARIANTS = {
    "Replace All": "local_replace_all_eval.json",
    "Replace One": "local_replace_one_eval.json",
    "Search":      "local_search_eval.json",
}
COLORS = {"Replace All": "#1f77b4", "Replace One": "#ff7f0e", "Search": "#2ca02c"}

# (json_key, title, y-axis label)
METRICS = [
    ("avg_pairwise_similarity", "Cosine similarity across runs", "avg pairwise cosine"),
    ("avg_pairwise_rougeL",     "ROUGE-L across runs",           "avg pairwise ROUGE-L"),
    ("avg_pairwise_tes",        "Token-edit similarity (TES)",   "avg pairwise TES"),
    ("same_answer_percentage",  "Same-answer % (LLM judge)",     "same-answer %"),
    ("unique_words",            "Unique words (round union)",    "unique words"),
    ("ai_reference_percentage", "AI-reference %",                "ai-reference %"),
]

def per_round(experiment, key):
    """Mean of metric `key` across questions per round (1-indexed on the x-axis when plotted)."""
    buckets = defaultdict(list)
    for q in experiment["questions"]:
        for it in q["iterations"]:
            v = it.get("metrics", {}).get(key)
            if v is not None:
                buckets[it["iteration_number"]].append(v)
    return [float(np.mean(buckets[r])) for r in sorted(buckets)]

def load_side(side):
    """Load the available variant eval files for one side model."""
    data = {}
    for label, fname in VARIANTS.items():
        p = os.path.join(EVAL_ROOT, side, fname)
        if os.path.isfile(p):
            with open(p, encoding="utf-8") as f:
                data[label] = json.load(f)
    return data

# ── Text-metric figures, one folder per side (RA/RO/Search overlaid) ──
for side in SIDES:
    data = load_side(side)
    if not data:
        print("skip (no eval files):", side); continue
    outdir = os.path.join(OUT_ROOT, side)
    os.makedirs(outdir, exist_ok=True)
    # one figure per metric
    for key, title, ylabel in METRICS:
        fig, ax = plt.subplots(figsize=(10, 6))
        for label, exp in data.items():
            vals = per_round(exp, key)
            if vals:
                ax.plot(range(1, len(vals) + 1), vals, linewidth=2.5, color=COLORS[label], label=label)
        ax.set_xlabel("Round", fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f"{side} — {title}", fontsize=13)
        ax.legend()
        fig.tight_layout()
        fig.savefig(f"{outdir}/{key}_per_round.png", dpi=150, bbox_inches="tight")
        plt.close(fig)
    # combined grid
    ncol = 2
    nrow = (len(METRICS) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(14, 5 * nrow))
    axes = np.array(axes).reshape(-1)
    for ax, (key, title, ylabel) in zip(axes, METRICS):
        for label, exp in data.items():
            vals = per_round(exp, key)
            if vals:
                ax.plot(range(1, len(vals) + 1), vals, linewidth=2.5, color=COLORS[label], label=label)
        ax.set_title(title, fontsize=13)
        ax.set_xlabel("Round", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.legend(fontsize=8)
    for ax in axes[len(METRICS):]:
        ax.set_visible(False)
    fig.suptitle(f"Cross-model baseline ({side}) — text metrics per round", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    fig.savefig(f"{outdir}/all_metrics_per_round.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"wrote {side}/: {list(data)}  ({len(METRICS)} metric PNGs + grid)")

wrote deepseek-r1-distill-qwen-7b/: ['Replace All', 'Replace One', 'Search']  (6 metric PNGs + grid)


wrote llama-3.1-8b/: ['Replace All', 'Replace One', 'Search']  (6 metric PNGs + grid)


wrote mistral-7b/: ['Replace All', 'Replace One', 'Search']  (6 metric PNGs + grid)


## Entity-collapse diagrams (all side models)

Cross-model entity data (Qwen2.5-14B main model, gpt-5.4-mini extraction + gpt-5.2 canonicalization) for
**all three side writers** and all three variants, read from the gitignored `cross_model_entity_logs/`
(`model_collapse_log_xm_<side>_<variant>.entities_by_round.jsonl`). Recomputed here (matching
`entity_extraction.py` / `visualization.ipynb`) and written into each side's folder,
`cross_model_baseline_visualizations/<side>/`:

- `unique_entities_per_round.png`: union of canonical entities across the 10 runs, averaged over questions
- `entity_similarity_per_round.png`: mean pairwise cosine similarity of binary entity-mention vectors
- `collapse_by_simulation.png`: % of question-rounds where all 10 runs share one entity set (95% Wilson CI)


In [ ]:
# ── Cross-model entity diagrams: per side (unique entities / entity similarity / collapse-by-simulation) ──
# Source: model_collapse_log_xm_<side>_<variant>.entities_by_round.jsonl in CROSS_ENTITY_DIR
# (gpt-5.4-mini extraction + gpt-5.2 canonicalization; Qwen2.5-14B answers each side model's documents).
# The JSONLs are large and gitignored; the PNGs written below are committed.
import re
from scipy.stats import binomtest

CROSS_ENTITY_DIR = "cross_model_entity_logs"
VLIST = ["Replace All", "Replace One", "Search"]
VKEY = {"Replace All": "replace_all", "Replace One": "replace_one", "Search": "search"}

def _entity_similarity(vectors):
    n = len(vectors)
    if n < 2:
        return 0.0
    s = []
    for i in range(n):
        for j in range(i + 1, n):
            a, b = vectors[i], vectors[j]
            na, nb = np.linalg.norm(a), np.linalg.norm(b)
            s.append(float(np.dot(a, b) / (na * nb)) if na > 0 and nb > 0 else 0.0)
    return float(np.mean(s))

def analyze_entity_file(path):
    """Per-round mean unique-entity count and mean entity similarity (matches entity_extraction.py)."""
    pru, prs, mx = defaultdict(list), defaultdict(list), 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            rounds, counts, mapped = rec["round_numbers"], rec["entity_counts"], rec["mapped_entities"]
            mx = max(mx, len(rounds))
            for idx in range(len(rounds)):
                pru[idx].append(len(counts[idx]))
                runs = mapped[idx]
                vocab = sorted({e for run in runs for e in run})
                vi = {e: i for i, e in enumerate(vocab)}
                vecs = []
                for run in runs:
                    v = np.zeros(len(vocab))
                    for e in run:
                        v[vi[e]] = 1.0
                    vecs.append(v)
                prs[idx].append(_entity_similarity(vecs))
    return ([float(np.mean(pru[i])) for i in range(mx)], [float(np.mean(prs[i])) for i in range(mx)])

def _wilson(count, nobs):
    if nobs == 0:
        return 0.0, 0.0, 0.0
    count, nobs = int(round(count)), int(round(nobs))
    ci = binomtest(count, nobs).proportion_ci(confidence_level=0.95, method="wilson")
    return 100.0 * count / nobs, 100.0 * ci.low, 100.0 * ci.high

def collapse_metrics(path):
    """% of question-rounds collapsed (all 10 runs share one canonical entity set), with 95% Wilson CIs."""
    n = ns = ne = cr = tr = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            mapped = rec["mapped_entities"]
            nr = len(rec["round_numbers"])
            flags = [len({frozenset(run) for run in mapped[i]}) == 1 for i in range(nr)]
            if not flags:
                continue
            n += 1; ns += int(flags[0]); ne += int(flags[-1]); cr += sum(flags); tr += len(flags)
    return {"start": _wilson(ns, n), "end": _wilson(ne, n), "rounds": _wilson(cr, tr)}

def _xm_entity_path(side, variant):
    return os.path.join(CROSS_ENTITY_DIR, f"model_collapse_log_xm_{side}_{VKEY[variant]}.entities_by_round.jsonl")

for side in SIDES:
    ent = {}
    for label in VLIST:
        path = _xm_entity_path(side, label)
        if not os.path.exists(path):
            print("  MISSING:", os.path.basename(path)); continue
        u, s = analyze_entity_file(path)
        ent[label] = {"color": COLORS[label], "unique": u, "similarity": s, "collapse": collapse_metrics(path)}
    if not ent:
        print("skip (no cross-model entity files):", side); continue
    outdir = os.path.join(OUT_ROOT, side); os.makedirs(outdir, exist_ok=True)

    for which, ylabel, title, fn in [
        ("unique", "Unique Entities", "Unique Entities Per Round", "unique_entities_per_round.png"),
        ("similarity", "Entity Similarity", "Entity Similarity Per Round", "entity_similarity_per_round.png")]:
        fig, ax = plt.subplots(figsize=(10, 6))
        for label, d in ent.items():
            vals = d[which]
            ax.plot(range(1, len(vals) + 1), vals, linewidth=2.5, label=label, color=d["color"])
        ax.set_xlabel("Round", fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f"{side} — {title}", fontsize=13)
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(outdir, fn), dpi=150, bbox_inches="tight")
        plt.close(fig)

    sims = list(ent.keys())
    x = np.arange(len(sims)); width = 0.25
    fig, ax = plt.subplots(figsize=(8, 6))
    for k, label, color, off in [
        ("start",  "% Collapsed at Start", "#1f77b4", -width),
        ("end",    "% Collapsed at End",   "#ff7f0e", 0.0),
        ("rounds", "% Rounds Collapsed",   "#2ca02c", width)]:
        p  = np.array([ent[s]["collapse"][k][0] for s in sims])
        lo = np.array([ent[s]["collapse"][k][1] for s in sims])
        hi = np.array([ent[s]["collapse"][k][2] for s in sims])
        yerr = np.vstack([p - lo, hi - p])
        bars = ax.bar(x + off, p, width, yerr=yerr, capsize=4, label=label, color=color)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f"{bar.get_height():.1f}", ha="center", va="bottom",
                    fontsize=9, fontweight="bold", color=color)
    ax.set_ylabel("%", fontsize=12)
    ax.set_title(f"{side} — Collapse by Simulation (error bars: 95% Wilson CI)", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s.lower()}\nsimulation" for s in sims], fontsize=10)
    ax.legend(loc="upper right")
    ax.set_ylim(0, 115)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "collapse_by_simulation.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("wrote entity figures:", side, list(ent))


## Side-model comparison — all side models overlaid, per metric

For each text metric, three subplots (replace_all / replace_one / search); within each, the **three side
models are overlaid** so the effect of *who wrote the documents* is read directly (shared y-axis per metric).
Saved to `cross_model_baseline_visualizations/_side_comparison/`.

In [3]:
CMP_OUT = os.path.join(OUT_ROOT, "_side_comparison"); os.makedirs(CMP_OUT, exist_ok=True)
SIDE_COLORS = {"deepseek-r1-distill-qwen-7b": "#1f77b4", "llama-3.1-8b": "#d62728", "mistral-7b": "#2ca02c"}
SIDE_LABEL  = {"deepseek-r1-distill-qwen-7b": "DeepSeek-R1-7B", "llama-3.1-8b": "Llama-3.1-8B", "mistral-7b": "Mistral-7B"}
VLIST = ["Replace All", "Replace One", "Search"]
alldata = {s: load_side(s) for s in SIDES}
for key, title, ylabel in METRICS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
    for ax, variant in zip(axes, VLIST):
        for s in SIDES:
            exp = alldata[s].get(variant)
            if not exp:
                continue
            vals = per_round(exp, key)
            if vals:
                ax.plot(range(1, len(vals) + 1), vals, linewidth=2.2, color=SIDE_COLORS[s], label=SIDE_LABEL[s])
        ax.set_title(variant, fontsize=12); ax.set_xlabel("Round", fontsize=10)
        ax.legend(fontsize=8)
    axes[0].set_ylabel(ylabel, fontsize=11)
    fig.suptitle(f"Side-model comparison — {title}", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(f"{CMP_OUT}/cmp_{key}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("wrote", f"{CMP_OUT}/cmp_{key}.png")

wrote cross_model_baseline_visualizations\_side_comparison/cmp_avg_pairwise_similarity.png


wrote cross_model_baseline_visualizations\_side_comparison/cmp_avg_pairwise_rougeL.png


wrote cross_model_baseline_visualizations\_side_comparison/cmp_avg_pairwise_tes.png


wrote cross_model_baseline_visualizations\_side_comparison/cmp_same_answer_percentage.png


wrote cross_model_baseline_visualizations\_side_comparison/cmp_unique_words.png


wrote cross_model_baseline_visualizations\_side_comparison/cmp_ai_reference_percentage.png


## Writer comparison: Qwen-self vs each foreign writer (self-preference control)

Overlays Qwen2.5-14B answering its **own** documents (from the gitignored entity re-run dump) against Qwen
answering each **foreign** writer's documents, per variant. This is the self-preference control: the
same-model baseline sits within the cross-model spread rather than above it. Saved to
`cross_model_baseline_visualizations/_writer_comparison/`.


In [ ]:
# ── Writer comparison: Qwen-self vs each foreign writer (self-preference control) ──
# Overlays Qwen2.5-14B answering its OWN documents (from the gitignored entity re-run dump) against Qwen
# answering each foreign writer's documents. Requires cell above (helpers, _xm_entity_path, VLIST, VKEY).
WCMP_OUT = os.path.join(OUT_ROOT, "_writer_comparison"); os.makedirs(WCMP_OUT, exist_ok=True)

_PAT = re.compile(r"entity re-run for workshop paper-.*")
_outer = sorted(d for d in os.listdir(".") if _PAT.fullmatch(d) and os.path.isdir(d))
if not _outer:
    print("skip writer comparison: no 'entity re-run for workshop paper-*' dump for the Qwen-self baseline")
else:
    _inner = [d for d in os.listdir(_outer[-1]) if os.path.isdir(os.path.join(_outer[-1], d))]
    SELF_BASE = os.path.join(_outer[-1], _inner[0]) if _inner else _outer[-1]

    WRITERS = [("self", "Qwen (self)", "#111111"),
               ("deepseek-r1-distill-qwen-7b", "DeepSeek-R1-7B", "#1f77b4"),
               ("llama-3.1-8b", "Llama-3.1-8B", "#d62728"),
               ("mistral-7b", "Mistral-7B", "#2ca02c")]

    def _writer_path(w, variant):
        if w == "self":
            v = VKEY[variant]
            return os.path.join(
                SELF_BASE,
                f"model_collapse_log_graphite_baseline_{v}_Qwen_Qwen2.5-14B-Instruct_local_{v}.entities_by_round.jsonl")
        return _xm_entity_path(w, variant)

    ANA, COL, ok = {}, {}, True
    for w, _, _ in WRITERS:
        for v in VLIST:
            p = _writer_path(w, v)
            if not os.path.exists(p):
                print("  MISSING:", os.path.basename(p)); ok = False; continue
            ANA[(w, v)] = analyze_entity_file(p)
            COL[(w, v)] = collapse_metrics(p)

    if ok:
        for idx, ylabel, fn in [(0, "Unique Entities", "cmp_unique_entities.png"),
                                (1, "Entity Similarity", "cmp_entity_similarity.png")]:
            fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
            for ax, v in zip(axes, VLIST):
                for w, wl, wc in WRITERS:
                    vals = ANA[(w, v)][idx]
                    ax.plot(range(1, len(vals) + 1), vals, color=wc, label=wl,
                            linewidth=2.5 if w == "self" else 2.0,
                            linestyle="-" if w == "self" else "--")
                ax.set_title(v, fontsize=12); ax.set_xlabel("Round", fontsize=10)
                ax.legend(fontsize=8)
            axes[0].set_ylabel(ylabel, fontsize=11)
            fig.suptitle(f"Writer comparison (Qwen answers each writer's docs) — {ylabel} Per Round", fontsize=14)
            fig.tight_layout(rect=[0, 0, 1, 0.96])
            fig.savefig(os.path.join(WCMP_OUT, fn), dpi=120, bbox_inches="tight")
            plt.close(fig)

        fig, ax = plt.subplots(figsize=(9, 5.2))
        x = np.arange(len(VLIST)); bw = 0.8 / len(WRITERS)
        for i, (w, wl, wc) in enumerate(WRITERS):
            p  = [COL[(w, v)]["end"][0] for v in VLIST]
            lo = [COL[(w, v)]["end"][0] - COL[(w, v)]["end"][1] for v in VLIST]
            hi = [COL[(w, v)]["end"][2] - COL[(w, v)]["end"][0] for v in VLIST]
            ax.bar(x + i * bw - 0.4 + bw / 2, p, bw, yerr=[lo, hi], capsize=3, label=wl, color=wc,
                   edgecolor="black" if w == "self" else "none", linewidth=0.8)
        ax.set_xticks(x); ax.set_xticklabels(VLIST); ax.set_ylabel("% Collapsed at End", fontsize=12)
        ax.set_ylim(0, 100); ax.legend(fontsize=9, ncol=2, frameon=False); ax.grid(axis="y", alpha=0.3)
        ax.set_title("Collapse by writer (error bars: 95% Wilson CI)", fontsize=12)
        fig.tight_layout()
        fig.savefig(os.path.join(WCMP_OUT, "cmp_collapse_end_by_writer.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)
        print("wrote writer comparison:", sorted(os.listdir(WCMP_OUT)))
